# Module 04: Sellers EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_sellers_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

sellers = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {sellers.shape}")
display(sellers.head(3))

Loaded raw dataset shape: (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [2]:
audit_df = pd.DataFrame(
    {
        "Data Type": sellers.dtypes,
        "Missing Values": sellers.isna().sum(),
        "Missing %": (sellers.isna().mean() * 100).round(2),
        "Unique Values": sellers.nunique(),
    }
)
display(audit_df)


,Data Type,Missing Values,Missing %,Unique Values
seller_id,object,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,object,0,0.0,611
seller_state,object,0,0.0,23


### 3. Data Type Optimization & Silver Layer Initialization

To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Identifiers (`string`):** Cast `seller_id` to explicit string type for reliable relational joins.
- **Zip Code Prefix (`string` with `zfill(5)`):** Preserves 5-digit formatting with zero-padding and enforces explicit string type to guarantee exact matching during geolocation joins.
- **City & State (`string` / `category`):** Standardizes text fields and casts state codes to categorical types to optimize memory overhead and improve filtering speed.

In [3]:
sellers_clean = sellers.copy()

sellers_clean["seller_id"] = sellers_clean["seller_id"].astype("string")
sellers_clean["seller_zip_code_prefix"] = sellers_clean["seller_zip_code_prefix"].astype(str).str.zfill(5).astype("string")
sellers_clean["seller_city"] = sellers_clean["seller_city"].astype("string")
sellers_clean["seller_state"] = sellers_clean["seller_state"].astype("category")

display(sellers_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
seller_id,string[python]
seller_zip_code_prefix,string[python]
seller_city,string[python]
seller_state,category


### 4. Text Standardization & Normalization

To eliminate discrepancies caused by formatting inconsistencies across geographical and identifier fields, we standardize all text columns:
- **Identifier Casting:** Convert `seller_id` into an explicit string type for reliable relational joins.
- **String Cleaning & Sanitization:** Replace placeholder variations with `NaN`, strip unwanted whitespaces, and enforce proper capitalization for city names (`.title()`).
- **State Code Harmonization:** Ensure all state abbreviations conform strictly to uppercase ISO standards and cast to `category`.
- **ZIP Code Formatting:** Convert postal code prefixes into 5-digit zero-padded strings to preserve leading zeros.

In [4]:
placeholders = ["", "-", "n/a", "null", "undefined", "unknown", "nan"]

sellers_clean["seller_id"] = sellers_clean["seller_id"].astype("string")

sellers_clean["seller_city"] = sellers_clean["seller_city"].replace(placeholders, np.nan)
sellers_clean["seller_city"] = sellers_clean["seller_city"].str.split().str.join(" ").str.title().astype("string")

sellers_clean["seller_state"] = sellers_clean["seller_state"].astype(str).str.strip().str.upper().astype("category")

sellers_clean["seller_zip_code_prefix"] = sellers_clean["seller_zip_code_prefix"].astype(str).str.zfill(5).astype("string")

display(sellers_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
seller_id,string[python]
seller_zip_code_prefix,string[python]
seller_city,string[python]
seller_state,category


In [5]:
sellers_clean.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

### 5. Data Integrity & Anomaly Audit

To ensure data quality prior to finalizing the Silver layer, we execute key validation checks:
- **ZIP Code Integrity:** Screening for negative or zero values in postal code prefixes (`seller_zip_code_prefix`).
- **State Code Validation:** Verifying that all state abbreviations strictly conform to the 2-character ISO format (`seller_state`).
- **Text Hygiene:** Inspecting city names (`seller_city`) for missing values (`NaN`) or empty strings.

In [6]:
anomaly_checks = {
    "Negative or zero zip code": 
        sellers_clean["seller_zip_code_prefix"].astype(int) <= 0,
    "Invalid state code length (!= 2 chars)": 
        ~sellers_clean["seller_state"].str.len().eq(2),
    "Empty city names": 
        sellers_clean["seller_city"].isna() | (sellers_clean["seller_city"].str.strip() == "")
}

for label, condition in anomaly_checks.items():
    print(f"{label}: {condition.sum()}")

Negative or zero zip code: 0
Invalid state code length (!= 2 chars): 0
Empty city names: 0


### 6. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [7]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {sellers.shape}")
print(f"Cleaned shape (Silver): {sellers_clean.shape}")
print(f"Total rows dropped: {len(sellers) - len(sellers_clean):,}")

dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': sellers.dtypes,
    'Silver (Clean)': sellers_clean.dtypes
})

print("Data Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (3095, 4)
Cleaned shape (Silver): (3095, 4)
Total rows dropped: 0
Data Types Transformation:


,Bronze (Raw),Silver (Clean)
seller_id,object,string[python]
seller_zip_code_prefix,int64,string[python]
seller_city,object,string[python]
seller_state,object,category


## 7. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to seller identity and geographic integrity:

* **Primary Key Integrity:** Ensure `seller_id` is entirely unique and free of missing values.
* **Mandatory Attributes:** Verify critical fields (`seller_zip_code_prefix`, `seller_city`, `seller_state`) are fully populated.
* **ZIP Code Integrity:** Confirm postal code prefixes contain no negative or zero anomalies and conform to valid ranges.
* **State Code Validation:** Verify all state abbreviations strictly conform to the 2-character ISO format.
* **Text Hygiene:** Ensure `seller_city` contains no missing values or empty strings.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [8]:
assert sellers_clean['seller_id'].is_unique, "Assertion failed: Duplicate seller_id found"
assert sellers_clean['seller_id'].notna().all(), "Assertion failed: NULL values found in seller_id"

critical_cols = ['seller_zip_code_prefix', 'seller_city', 'seller_state']
for col in critical_cols:
    assert sellers_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

zip_ints = sellers_clean['seller_zip_code_prefix'].astype(int)
invalid_zips = (zip_ints <= 0) | (zip_ints > 99999)
assert not invalid_zips.any(), "Assertion failed: Negative, zero, or out-of-range ZIP code prefix found"

invalid_states = ~sellers_clean['seller_state'].str.len().eq(2)
assert not invalid_states.any(), "Assertion failed: Invalid state code length found"

invalid_cities = sellers_clean['seller_city'].isna() | (sellers_clean['seller_city'].str.strip() == "")
assert not invalid_cities.any(), "Assertion failed: Empty or missing seller_city values found"

assert len(sellers_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
sellers_clean.to_parquet("../data/sellers_clean.parquet", index=False)
print("Sellers dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Sellers dataset successfully saved to Parquet.
